## WISDM Data Preprocessing notebook
### Step 0. Setup the paths and env variables

In [1]:
# =========================
# STEP 0 — Setup & contracts
# =========================
from pathlib import Path
import json, sys, re
import numpy as np
import pandas as pd
from tqdm import tqdm

ROOT = Path("/home/aidan/IMU_LM_Data")
sys.path.insert(0, str(ROOT))

from UTILS.helpers import upsample_df_rate, nearest_label_join_1d, _canon

BASE      = ROOT / "data"
RAW_WISDM = BASE / "raw_data" / "WISDM" / "watch"
CLEANED   = BASE / "cleaned_premerge"

SCHEMA_PATH       = ROOT / "Unification" / "schemas" / "continuous_stream_schema.json"
ACTIVITY_MAP_PATH = ROOT / "Unification" / "schemas" / "activity_mapping.json"

SCHEMA       = json.loads(SCHEMA_PATH.read_text())
ACT_MAP_FULL = json.loads(ACTIVITY_MAP_PATH.read_text())

UNKNOWN_ID = int(ACT_MAP_FULL.get("unknown_activity_id", 9000))
TARGET_HZ  = int(SCHEMA.get("rate_hz", 50))

RAW2ID  = { _canon(k): int(v) for k, v in ACT_MAP_FULL.get("mapping", {}).items() }
ID2NAME = { int(x["id"]): x["name"] for x in ACT_MAP_FULL["label_set"] }

ACC_DIR  = RAW_WISDM / "acc"
GYRO_DIR = RAW_WISDM / "gyro"

print("RAW_WISDM:", RAW_WISDM)
print("ACC_DIR :", ACC_DIR)
print("GYRO_DIR:", GYRO_DIR)
print("TARGET_HZ:", TARGET_HZ)


RAW_WISDM: /home/aidan/IMU_LM_Data/data/raw_data/WISDM/watch
ACC_DIR : /home/aidan/IMU_LM_Data/data/raw_data/WISDM/watch/acc
GYRO_DIR: /home/aidan/IMU_LM_Data/data/raw_data/WISDM/watch/gyro
TARGET_HZ: 50


### Step 1. Ingest, preporccess and map the data 

In [ ]:
# ============================
# STEP 1 — Load, normalize, resample → 50Hz (native labels aligned)
#   FIX: WISDM timestamps are not globally monotonic across activities.
#        Split into per-(subject, activity) segments and resample each segment independently.
# ============================

WISDM_CODE2LABEL = {
    "A": "walking",
    "B": "jogging",
    "C": "stairs",
    "D": "sitting",
    "E": "standing",
    "F": "typing",
    "G": "brushing_teeth",
    "H": "eating_soup",
    "I": "eating_chips",
    "J": "eating_pasta",
    "K": "drinking_from_cup",
    "L": "eating_sandwich",
    "M": "kicking_soccer_ball",
    "O": "playing_catch_tennis_ball",
    "P": "dribbling_basketball",
    "Q": "writing",
    "R": "clapping",
    "S": "folding_clothes",
}
WISDM_CODE2ID = {c: i + 1 for i, c in enumerate(sorted(WISDM_CODE2LABEL.keys()))}

def _fid(p: Path):
    m = re.search(r"data_(\d+)_", p.name)
    return m.group(1) if m else None

def _parse_wisdm_file(path: Path) -> pd.DataFrame:
    """
    Format per line:
      subject, activity_code, timestamp, x, y, z;
    Timestamp is Linux time in *nanoseconds* per dataset doc.
    """
    rows = []
    for ln in path.read_text().splitlines():
        ln = ln.strip()
        if not ln:
            continue
        if ln.endswith(";"):
            ln = ln[:-1]
        parts = [p.strip() for p in ln.split(",")]
        if len(parts) != 6:
            continue
        sid, code, ts, x, y, z = parts
        rows.append((sid, code, int(ts), float(x), float(y), float(z)))
    return pd.DataFrame(rows, columns=["subject_id", "code", "timestamp_abs", "x", "y", "z"])

def _resample_session_to_50hz(df_sess: pd.DataFrame, src_hz: float = 20.0) -> pd.DataFrame:
    """
    df_sess columns:
      dataset, subject_id, session_id, timestamp_abs, acc_x/y/z, gyro_x/y/z,
      dataset_activity_id, dataset_activity_label
    Output:
      same but timestamp_ns is session-relative and resampled @50Hz.
    """
    df_sess = df_sess.sort_values("timestamp_abs", kind="mergesort").reset_index(drop=True)
    if df_sess.empty:
        return df_sess

    # session-relative ns (WISDM timestamps are already ns)
    t0 = int(df_sess["timestamp_abs"].min())
    ts_ns = (df_sess["timestamp_abs"].astype("int64") - t0).astype("int64")
    ts_s  = ts_ns.to_numpy(dtype=np.float64) / 1e9

    NUM    = ["acc_x", "acc_y", "acc_z", "gyro_x", "gyro_y", "gyro_z"]
    LABELS = ["dataset_activity_id", "dataset_activity_label"]

    tmp = df_sess[NUM].copy()
    tmp.insert(0, "timestamp_s", ts_s)


    up = upsample_df_rate(
        df=tmp,
        tcol="timestamp_s",
        num_cols=NUM,
        src_hz=src_hz,
        dst_hz=TARGET_HZ,  # 50
    )

    up["timestamp_ns"] = (up["timestamp_s"] * 1e9).round().astype("int64")

    # labels are constant within this segment, but keep your general alignment mechanism
    HALF_FRAME_NS = int(1e9 / (2 * TARGET_HZ))  # 10ms
    lab_src = pd.DataFrame({
        "timestamp_ns": ts_ns,
        "dataset_activity_id": df_sess["dataset_activity_id"].astype("int16"),
        "dataset_activity_label": df_sess["dataset_activity_label"].astype("string"),
    })

    aligned = nearest_label_join_1d(
        src_ts_ns=lab_src["timestamp_ns"].to_numpy(),
        src_label_df=lab_src[LABELS].copy(),
        target_ts_ns=up["timestamp_ns"].to_numpy(),
        half_frame_ns=HALF_FRAME_NS,
    ).ffill().bfill()

    out = pd.DataFrame({
        "dataset": df_sess["dataset"].iloc[0],
        "subject_id": df_sess["subject_id"].iloc[0],
        "session_id": df_sess["session_id"].iloc[0],
        "timestamp_ns": up["timestamp_ns"].astype("int64"),

        "acc_x": up["acc_x"].astype("float32"),
        "acc_y": up["acc_y"].astype("float32"),
        "acc_z": up["acc_z"].astype("float32"),
        "gyro_x": up["gyro_x"].astype("float32"),
        "gyro_y": up["gyro_y"].astype("float32"),
        "gyro_z": up["gyro_z"].astype("float32"),

        "dataset_activity_id": aligned["dataset_activity_id"].astype("int16"),
        "dataset_activity_label": aligned["dataset_activity_label"].astype("string"),
    })
    return out

def load_wisdm_watch_resampled(acc_dir: Path, gyro_dir: Path) -> pd.DataFrame:
    acc_files  = sorted(acc_dir.glob("data_*_acc*watch*"))
    gyro_files = sorted(gyro_dir.glob("data_*_gyro*watch*"))

    acc_map  = {_fid(p): p for p in acc_files if _fid(p) is not None}
    gyro_map = {_fid(p): p for p in gyro_files if _fid(p) is not None}
    fids = sorted(set(acc_map.keys()) & set(gyro_map.keys()))
    if not fids:
        print("No matched acc/gyro session files found.")
        return pd.DataFrame()

    out_sessions = []
    for fid in tqdm(fids, desc="[WISDM] load+split+resample segments"):
        acc = _parse_wisdm_file(acc_map[fid])
        gyr = _parse_wisdm_file(gyro_map[fid])

        m = acc.merge(
            gyr,
            on=["subject_id", "code", "timestamp_abs"],
            how="inner",
            suffixes=("_acc", "_gyro"),
        )
        if m.empty:
            continue

        # native label/id
        m["dataset_activity_label"] = m["code"].map(WISDM_CODE2LABEL).fillna("unknown_activity")
        m["dataset_activity_id"] = m["code"].map(WISDM_CODE2ID).fillna(UNKNOWN_ID).astype("int16")

        # axis mapping (your convention)
        m["acc_x"]  = m["y_acc"]
        m["acc_y"]  = m["x_acc"]
        m["acc_z"]  = m["z_acc"]
        m["gyro_x"] = m["y_gyro"]
        m["gyro_y"] = m["x_gyro"]
        m["gyro_z"] = m["z_gyro"]

        # IMPORTANT FIX:
        # WISDM time is not globally monotonic across activities, so split by activity code.
        # Each (subject=fid, code) is a separate ~3min segment.
        for code, g in m.groupby("code", sort=False):
            sess = g[[
                "subject_id", "timestamp_abs",
                "acc_x", "acc_y", "acc_z",
                "gyro_x", "gyro_y", "gyro_z",
                "dataset_activity_id", "dataset_activity_label",
            ]].copy()

            sess.insert(0, "dataset", "wisdm")
            # unique segment session id: "<subject>_<code>"
            sess.insert(2, "session_id", f"{fid}_{code}")

            out_sessions.append(_resample_session_to_50hz(sess, src_hz=20.0))

    out = pd.concat(out_sessions, ignore_index=True) if out_sessions else pd.DataFrame()
    print("WISDM rows @50Hz:", len(out))
    return out

wisdm_50_native = load_wisdm_watch_resampled(ACC_DIR, GYRO_DIR)
raw_acc  = sum(len(_parse_wisdm_file(p)) for p in ACC_DIR.glob("data_*_acc*watch*"))
raw_gyro = sum(len(_parse_wisdm_file(p)) for p in GYRO_DIR.glob("data_*_gyro*watch*"))

print(f"PRE  (raw acc rows):  {raw_acc:,}")
print(f"PRE  (raw gyro rows): {raw_gyro:,}")
print(f"POST (50Hz rows):     {len(wisdm_50_native):,}")

wisdm_50_native.head(3)


[WISDM] load+split+resample segments:   2%|▏         | 1/51 [00:00<00:12,  4.05it/s]

[DEBUG RAW] session=1600_A label=walking raw_rows=3,603 dur_s=179.75 expected_50hz≈8,988
[DEBUG RAW] session=1600_B label=jogging raw_rows=3,604 dur_s=179.81 expected_50hz≈8,991
[DEBUG RAW] session=1600_C label=stairs raw_rows=3,604 dur_s=179.79 expected_50hz≈8,991
[DEBUG RAW] session=1600_D label=sitting raw_rows=3,604 dur_s=179.81 expected_50hz≈8,991
[DEBUG RAW] session=1600_E label=standing raw_rows=3,604 dur_s=179.90 expected_50hz≈8,996
[DEBUG RAW] session=1600_F label=typing raw_rows=3,604 dur_s=179.90 expected_50hz≈8,996
[DEBUG RAW] session=1600_G label=brushing_teeth raw_rows=3,603 dur_s=179.78 expected_50hz≈8,990
[DEBUG RAW] session=1600_H label=eating_soup raw_rows=3,603 dur_s=179.76 expected_50hz≈8,989
[DEBUG RAW] session=1600_I label=eating_chips raw_rows=3,604 dur_s=179.80 expected_50hz≈8,991
[DEBUG RAW] session=1600_J label=eating_pasta raw_rows=3,603 dur_s=179.78 expected_50hz≈8,990
[DEBUG RAW] session=1600_K label=drinking_from_cup raw_rows=3,604 dur_s=179.80 expected_50

[WISDM] load+split+resample segments:   4%|▍         | 2/51 [00:00<00:11,  4.26it/s]

[DEBUG RAW] session=1601_A label=walking raw_rows=3,601 dur_s=179.87 expected_50hz≈8,995
[DEBUG RAW] session=1601_B label=jogging raw_rows=3,602 dur_s=179.91 expected_50hz≈8,997
[DEBUG RAW] session=1601_C label=stairs raw_rows=3,601 dur_s=179.87 expected_50hz≈8,995
[DEBUG RAW] session=1601_D label=sitting raw_rows=3,601 dur_s=179.88 expected_50hz≈8,995
[DEBUG RAW] session=1601_E label=standing raw_rows=3,602 dur_s=179.94 expected_50hz≈8,998
[DEBUG RAW] session=1601_F label=typing raw_rows=3,601 dur_s=179.88 expected_50hz≈8,995
[DEBUG RAW] session=1601_G label=brushing_teeth raw_rows=3,602 dur_s=179.93 expected_50hz≈8,998
[DEBUG RAW] session=1601_H label=eating_soup raw_rows=3,602 dur_s=179.92 expected_50hz≈8,997
[DEBUG RAW] session=1601_I label=eating_chips raw_rows=3,602 dur_s=179.94 expected_50hz≈8,998
[DEBUG RAW] session=1601_J label=eating_pasta raw_rows=3,602 dur_s=179.93 expected_50hz≈8,997
[DEBUG RAW] session=1601_K label=drinking_from_cup raw_rows=3,601 dur_s=179.87 expected_50

[WISDM] load+split+resample segments:   6%|▌         | 3/51 [00:00<00:11,  4.35it/s]

...

[WISDM] load+split+resample segments:  43%|████▎     | 22/51 [00:05<00:06,  4.25it/s]

[DEBUG RAW] session=1621_A label=walking raw_rows=3,600 dur_s=179.74 expected_50hz≈8,988
[DEBUG RAW] session=1621_B label=jogging raw_rows=3,600 dur_s=179.74 expected_50hz≈8,988
[DEBUG RAW] session=1621_C label=stairs raw_rows=3,600 dur_s=179.75 expected_50hz≈8,988
[DEBUG RAW] session=1621_D label=sitting raw_rows=7,918 dur_s=179.84 expected_50hz≈8,993
[DEBUG RAW] session=1621_E label=standing raw_rows=5,816 dur_s=179.77 expected_50hz≈8,989
[DEBUG RAW] session=1621_F label=typing raw_rows=3,600 dur_s=179.75 expected_50hz≈8,988
[DEBUG RAW] session=1621_G label=brushing_teeth raw_rows=3,602 dur_s=179.84 expected_50hz≈8,993
[DEBUG RAW] session=1621_H label=eating_soup raw_rows=3,600 dur_s=179.75 expected_50hz≈8,988
[DEBUG RAW] session=1621_I label=eating_chips raw_rows=3,600 dur_s=179.75 expected_50hz≈8,988
[DEBUG RAW] session=1621_J label=eating_pasta raw_rows=3,600 dur_s=179.74 expected_50hz≈8,988
[DEBUG RAW] session=1621_K label=drinking_from_cup raw_rows=3,600 dur_s=179.74 expected_50

[WISDM] load+split+resample segments:  45%|████▌     | 23/51 [00:05<00:06,  4.36it/s]

[DEBUG RAW] session=1622_A label=walking raw_rows=3,603 dur_s=179.74 expected_50hz≈8,988
[DEBUG RAW] session=1622_B label=jogging raw_rows=3,603 dur_s=179.76 expected_50hz≈8,989
[DEBUG RAW] session=1622_C label=stairs raw_rows=3,604 dur_s=179.79 expected_50hz≈8,990
[DEBUG RAW] session=1622_D label=sitting raw_rows=3,603 dur_s=179.77 expected_50hz≈8,990
[DEBUG RAW] session=1622_E label=standing raw_rows=3,603 dur_s=179.79 expected_50hz≈8,991
[DEBUG RAW] session=1622_F label=typing raw_rows=3,603 dur_s=179.79 expected_50hz≈8,990
[DEBUG RAW] session=1622_G label=brushing_teeth raw_rows=3,603 dur_s=179.78 expected_50hz≈8,990
[DEBUG RAW] session=1622_H label=eating_soup raw_rows=3,603 dur_s=179.79 expected_50hz≈8,991
[DEBUG RAW] session=1622_I label=eating_chips raw_rows=3,603 dur_s=179.79 expected_50hz≈8,990
[DEBUG RAW] session=1622_J label=eating_pasta raw_rows=3,603 dur_s=179.78 expected_50hz≈8,990
[DEBUG RAW] session=1622_K label=drinking_from_cup raw_rows=3,603 dur_s=179.78 expected_50

KeyboardInterrupt: 

### Step 2. Map the data and audit the mapping

In [3]:
# ============================
# STEP 2 — Mapping audit (native → global)
# ============================
if wisdm_50_native.empty:
    raise SystemExit("No WISDM rows after STEP 1. Check raw paths / parsing.")

raw_counts = (
    wisdm_50_native["dataset_activity_label"]
    .astype("string")
    .map(_canon)
    .value_counts()
    .rename_axis("raw_label")
    .reset_index(name="count")
)

raw_counts["mapped_id"] = raw_counts["raw_label"].map(RAW2ID).fillna(UNKNOWN_ID).astype(int)
raw_counts["mapped_nm"] = raw_counts["mapped_id"].map(lambda x: ID2NAME.get(int(x), "other"))

unmapped = raw_counts.loc[raw_counts["mapped_id"] == UNKNOWN_ID]
print(f"[WISDM] Unique raw labels: {len(raw_counts)} | Unmapped: {len(unmapped)}")

total_ct = int(raw_counts["count"].sum())
mapped_ct = int(raw_counts.loc[raw_counts["mapped_id"] != UNKNOWN_ID, "count"].sum())
print(f"coverage={100.0*mapped_ct/max(total_ct,1):.2f}%  (mapped={mapped_ct:,}/{total_ct:,})")

if not unmapped.empty:
    print("\nUnmapped raw labels:")
    print(unmapped.sort_values("count", ascending=False).to_string(index=False))


KeyboardInterrupt: 

### Step 3. Build and clean dataset in stream json fromat

In [7]:
# ============================
# STEP 3 — Add global_* and schema-order
# ============================
def to_continuous_stream_wisdm(df_50_native: pd.DataFrame) -> pd.DataFrame:
    if df_50_native.empty:
        return pd.DataFrame(columns=[c["name"] for c in SCHEMA["columns"]])

    raw_key = df_50_native["dataset_activity_label"].astype("string").map(_canon)
    gid     = raw_key.map(RAW2ID).fillna(UNKNOWN_ID).astype("int16")
    glabel  = gid.map(lambda x: ID2NAME.get(int(x), "other")).astype("string")

    out = pd.DataFrame({
        "dataset":     df_50_native["dataset"].astype("string"),
        "subject_id":  df_50_native["subject_id"].astype("string"),
        "session_id":  df_50_native["session_id"].astype("string"),
        "timestamp_ns": df_50_native["timestamp_ns"].astype("int64"),

        "acc_x": df_50_native["acc_x"].astype("float32"),
        "acc_y": df_50_native["acc_y"].astype("float32"),
        "acc_z": df_50_native["acc_z"].astype("float32"),
        "gyro_x": df_50_native["gyro_x"].astype("float32"),
        "gyro_y": df_50_native["gyro_y"].astype("float32"),
        "gyro_z": df_50_native["gyro_z"].astype("float32"),

        "global_activity_id": gid,
        "global_activity_label": glabel,

        "dataset_activity_id": df_50_native["dataset_activity_id"].astype("int16"),
        "dataset_activity_label": df_50_native["dataset_activity_label"].astype("string"),
    })

    order = [c["name"] for c in SCHEMA["columns"]]
    return out[order]

wisdm_df = to_continuous_stream_wisdm(wisdm_50_native)
print("WISDM unified rows:", len(wisdm_df))
wisdm_df.head(3)


WISDM unified rows: 59980920


,dataset,subject_id,session_id,timestamp_ns,acc_x,acc_y,acc_z,gyro_x,gyro_y,gyro_z,global_activity_id,global_activity_label,dataset_activity_id,dataset_activity_label
0,wisdm,1600,1600,0,-1.054047,3.613748,11.779023,-1.544470,-1.365979,-1.696995,1,posture_stationary,4,sitting
1,wisdm,1600,1600,20000000,-2.014125,2.999722,12.260026,-1.433817,-1.583854,-1.745460,1,posture_stationary,4,sitting
2,wisdm,1600,1600,40000000,-2.974203,2.385696,12.741029,-1.323164,-1.801729,-1.793924,1,posture_stationary,4,sitting


### Step 4. Audit check the unified frame

In [8]:
# ============================
# STEP 4 — QA
# ============================
groups = ["subject_id","session_id"]

print("Subjects:", wisdm_df["subject_id"].nunique(), "| Sessions:", wisdm_df["session_id"].nunique())

# monotonic per group
viol = 0
for (_sid, _sess), g in wisdm_df.groupby(groups, sort=False):
    ts = g["timestamp_ns"].to_numpy()
    if ts.size and not np.all(np.diff(ts) >= 0):
        viol += 1
print("Monotonic violations (groups):", viol)

def est_hz_ns(ts_ns: pd.Series):
    arr = ts_ns.to_numpy()
    if arr.size < 3: 
        return np.nan
    dt = np.diff(arr) / 1e9
    dt = dt[(dt > 0) & np.isfinite(dt)]
    return float(np.median(1.0/dt)) if dt.size else np.nan

hz = wisdm_df.groupby(groups)["timestamp_ns"].apply(est_hz_ns)
print(f"Median Hz: {np.nanmedian(hz.values):.2f} (target={SCHEMA['rate_hz']})")

req = SCHEMA["expectations"]["required_not_null"]
pct = wisdm_df[req].notnull().all(axis=1).mean() * 100
print(f"Rows meeting required-not-null: {pct:.2f}%")

cov = (wisdm_df["global_activity_id"] != UNKNOWN_ID).mean() * 100
print(f"Global mapping coverage: {cov:.1f}% (unknown={UNKNOWN_ID})")

print("\nTop-15 global labels:")
print(wisdm_df["global_activity_label"].value_counts().head(15))


Subjects: 51 | Sessions: 51
Monotonic violations (groups): 0
Median Hz: 50.00 (target=50)
Rows meeting required-not-null: 100.00%
Global mapping coverage: 100.0% (unknown=9000)

Top-15 global labels:
global_activity_label
stairs                   27026399
run_jog                  12139758
sports_play               7684289
adl_food                  4985360
posture_stationary        3370360
adl_personal_care         2126030
adl_desk_device           1395909
adl_household_general      659286
walk                       593529
Name: count, dtype: Int64


In [ ]:
import numpy as np

# ---- subsample (e.g., 1 million rows max) ----
N = min(len(wisdm_df), 1_000_000)
sample = wisdm_df.sample(N, random_state=0)

# ---- accel magnitude ----
acc_mag = np.sqrt(
    sample["acc_x"]**2 +
    sample["acc_y"]**2 +
    sample["acc_z"]**2
)

print("ACC magnitude (m/s^2):")
print(acc_mag.describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]))

# ---- gyro magnitude ----
gyro_mag = np.sqrt(
    sample["gyro_x"]**2 +
    sample["gyro_y"]**2 +
    sample["gyro_z"]**2
)

print("\nGYRO magnitude (rad/s):")
print(gyro_mag.describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]))

dur_ns = (
    wisdm_df
    .groupby(["subject_id", "session_id"])["timestamp_ns"]
    .max()
)

dur_hours = dur_ns / 1e9 / 3600

print(dur_hours.describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.99]))

from pathlib import Path

def count_raw_wisdm_rows(acc_dir: Path, gyro_dir: Path):
    acc_rows = 0
    gyro_rows = 0

    for p in acc_dir.glob("data_*"):
        acc_rows += sum(1 for _ in open(p))

    for p in gyro_dir.glob("data_*"):
        gyro_rows += sum(1 for _ in open(p))

    return acc_rows, gyro_rows

acc_raw, gyro_raw = count_raw_wisdm_rows(ACC_DIR, GYRO_DIR)

print(f"Raw ACC rows : {acc_raw:,}")
print(f"Raw GYRO rows: {gyro_raw:,}")

# WISDM watch is ~20 Hz
EXPECTED_50HZ = int(acc_raw * (50 / 20))

print(f"Expected ~50Hz rows: {EXPECTED_50HZ:,}")
print(f"Actual   50Hz rows: {len(wisdm_df):,}")
print(f"Ratio actual / expected: {len(wisdm_df) / EXPECTED_50HZ:.3f}")





ACC magnitude (m/s^2):
count    1000000.000000
mean           9.834066
std            2.692114
min            0.324442
1%             4.823013
5%             5.335036
25%            8.232217
50%            9.652169
75%           11.476717
95%           14.161411
99%           15.972421
max           61.853428
dtype: float64

GYRO magnitude (rad/s):
count    1000000.000000
mean           1.515158
std            1.274606
min            0.000290
1%             0.016277
5%             0.162502
25%            0.795988
50%            1.001562
75%            2.155240
95%            3.804675
99%            6.384591
max           30.455622
dtype: float64


### Step 5. Save outputs

In [ ]:
# ============================
# STEP 5 — Save
# ============================
out_path = CLEANED / "wisdm_clean_data.parquet"
out_path.parent.mkdir(parents=True, exist_ok=True)
wisdm_df.to_parquet(out_path, index=False)
print("Saved →", out_path)
